In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
%matplotlib inline

## 1. Cargar y Explorar los Datos

In [ ]:
# Cargar el dataset de cáncer de mama
cancer = load_breast_cancer()
X = cancer.data  # Características
y = cancer.target  # Etiquetas (0: maligno, 1: benigno)

# Crear DataFrame
df = pd.DataFrame(X, columns=cancer.feature_names)
df['diagnosis'] = pd.Categorical.from_codes(y, ['maligno', 'benigno'])

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()

In [ ]:
# Información general del dataset
print("Información del dataset:")
print(f"Total de muestras: {len(df)}")
print(f"Total de características: {len(cancer.feature_names)}")
print(f"\nValores nulos: {df.isnull().sum().sum()}")

In [ ]:
# Distribución de clases
print("Distribución de diagnósticos:")
print(df['diagnosis'].value_counts())
print(f"\nPorcentaje:")
print(df['diagnosis'].value_counts(normalize=True) * 100)

# Visualización
plt.figure(figsize=(8, 5))
df['diagnosis'].value_counts().plot(kind='bar', color=['salmon', 'lightgreen'])
plt.title('Distribución de Diagnósticos')
plt.xlabel('Diagnóstico')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar la correlación de algunas características principales
main_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'diagnosis']
plt.figure(figsize=(10, 8))
sns.pairplot(df[main_features], hue='diagnosis', diag_kind='kde')
plt.suptitle('Relación entre Características Principales', y=1.01)
plt.tight_layout()
plt.show()

## 2. Preparar los Datos

In [ ]:
# Dividir los datos en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de prueba: {X_test.shape[0]} muestras")

In [ ]:
# Normalizar los datos (importante para regresión logística)
# La regresión logística es sensible a la escala de las características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos normalizados exitosamente")
print(f"Media de X_train_scaled: {X_train_scaled.mean():.6f}")
print(f"Desviación estándar de X_train_scaled: {X_train_scaled.std():.6f}")

## 3. Entrenar el Modelo de Regresión Logística

In [ ]:
# Crear el modelo de regresión logística
# max_iter: número máximo de iteraciones
# random_state: para reproducibilidad
log_reg = LogisticRegression(max_iter=10000, random_state=42)

# Entrenar el modelo
log_reg.fit(X_train_scaled, y_train)

print("Modelo entrenado exitosamente")
print(f"Número de iteraciones realizadas: {log_reg.n_iter_[0]}")

## 4. Realizar Predicciones

In [ ]:
# Hacer predicciones en el conjunto de prueba
y_pred = log_reg.predict(X_test_scaled)

# Obtener las probabilidades predichas
y_pred_proba = log_reg.predict_proba(X_test_scaled)

# Mostrar algunas predicciones con sus probabilidades
print("Ejemplos de predicciones con probabilidades:")
resultados = pd.DataFrame({
    'Real': ['maligno' if i == 0 else 'benigno' for i in y_test[:10]],
    'Predicho': ['maligno' if i == 0 else 'benigno' for i in y_pred[:10]],
    'Prob_Maligno': y_pred_proba[:10, 0],
    'Prob_Benigno': y_pred_proba[:10, 1]
})
print(resultados)

## 5. Evaluar el Modelo

In [ ]:
# Calcular métricas de evaluación
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Métricas de Evaluación:")
print(f"Precisión (Accuracy): {accuracy:.2%}")
print(f"Precisión (Precision): {precision:.2%}")
print(f"Sensibilidad (Recall): {recall:.2%}")
print(f"F1-Score: {f1:.4f}")

In [ ]:
# Reporte de clasificación detallado
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['maligno', 'benigno']))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['maligno', 'benigno'],
            yticklabels=['maligno', 'benigno'])
plt.title('Matriz de Confusión - Regresión Logística')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predicho')
plt.tight_layout()
plt.show()

## 6. Curva ROC y AUC

In [ ]:
# Calcular la curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba[:, 1])
roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])

# Visualizar la curva ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC - Regresión Logística')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nÁrea bajo la curva (AUC): {roc_auc:.4f}")

## 7. Validación Cruzada

In [ ]:
# Realizar validación cruzada con 5 pliegues
cv_scores = cross_val_score(log_reg, X_train_scaled, y_train, cv=5)

print("Scores de Validación Cruzada (5-fold):")
print(cv_scores)
print(f"\nMedia: {cv_scores.mean():.4f}")
print(f"Desviación estándar: {cv_scores.std():.4f}")

# Visualización
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_scores, color='steelblue', alpha=0.7)
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', label=f'Media: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Scores de Validación Cruzada')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Importancia de las Características

In [ ]:
# Obtener los coeficientes del modelo (importancia de características)
coefficients = pd.DataFrame({
    'Característica': cancer.feature_names,
    'Coeficiente': log_reg.coef_[0]
}).sort_values('Coeficiente', key=abs, ascending=False)

print("Top 10 características más importantes:")
print(coefficients.head(10))

# Visualización
plt.figure(figsize=(10, 8))
top_10 = coefficients.head(10)
colors = ['red' if x < 0 else 'green' for x in top_10['Coeficiente']]
plt.barh(top_10['Característica'], top_10['Coeficiente'], color=colors, alpha=0.7)
plt.xlabel('Coeficiente')
plt.title('Top 10 Características más Importantes\n(Rojo: Indica malignidad, Verde: Indica benignidad)')
plt.tight_layout()
plt.show()

## Conclusiones

- La regresión logística es un modelo eficiente y fácil de interpretar para clasificación binaria
- Proporciona probabilidades de pertenencia a cada clase, no solo predicciones
- La normalización de características es importante para el rendimiento del modelo
- Los coeficientes del modelo indican la contribución de cada característica a la predicción
- La curva ROC y el AUC son métricas importantes para evaluar modelos de clasificación